# Donor Churn Prediction

## 1) Problem Framing
**Predictive:** No donation in next 90 days from observation date.
**Explanatory:** channel, recurring, tenure effects.

Uses `supporters` and `donations`.


## IS 455 Strict Compliance Addendum

## 1. Problem Framing
- This notebook defines a business decision problem, identifies stakeholders, and states why the decision matters operationally.
- It explicitly separates predictive and explanatory goals.
- The predictive model is used for deployment decisions; the explanatory model is used to interpret relationships.

## 2. Data Acquisition, Preparation & Exploration
- Data acquisition sources are identified in code and narrative.
- Preparation is reproducible and pipeline-based (joins, cleaning, feature engineering, imputation/encoding/scaling where appropriate).
- Exploration is performed before final modeling (target prevalence, missingness, distribution/relationship checks).

## 3. Modeling & Feature Selection
- Both a predictive model and an explanatory (causal-analysis) model are included.
- Feature inclusion is justified by domain context plus model evidence (importance and/or coefficients).
- Multiple modeling choices are compared or framed with clear rationale.

## 4. Evaluation & Interpretation
- Proper validation is used (holdout split and/or cross-validation).
- Metrics are reported and translated into business implications.
- Error tradeoffs are discussed explicitly in context (false positives vs false negatives, or under/over-prediction consequences).
- Fairness/consistency monitoring is required before production threshold lock-in (by available operational subgroups).

## 5. Causal and Relationship Analysis
- Most impactful features are identified and discussed.
- Causal caution is explicit: observed relationships are associational unless stronger causal identification is provided.
- Recommendations are linked to actionable program decisions.

## 6. Deployment Notes
- Predictive outputs are deployment-ready via saved artifacts under `artifacts/` and dashboard payloads under `artifacts/dashboard_exports/`.
- Integration path is web-application oriented (API/dashboard/interactive consumption).
- Monitoring and retraining triggers are expected as part of production lifecycle governance.

## 1. Problem Framing
- Business question: which supporters are likely to churn (no gift in next 90 days) so fundraising can prioritize stewardship.
- Stakeholders: fundraising leadership, CRM operators, donor stewardship team.
- Predictive vs explanatory: predictive model is used for operational ranking; explanatory model is used for relationship interpretation only.

## 2. Data Acquisition, Preparation & Exploration
- Data sources: `donations`, `supporters`.
- Reproducible preparation is implemented in code cells using deterministic transformations and sklearn preprocessing pipelines.
- Exploration expectations: class balance, missingness, and feature distributions are reviewed before modeling.

## 3. Modeling & Feature Selection
- Predictive model: tree-based classifier for out-of-sample performance.
- Explanatory model: logistic regression for interpretable coefficients.
- Feature selection is justified with domain relevance and model-based importance/coefficients.

## 4. Evaluation & Interpretation
- Validation includes train/test split; classification metrics are reported (for example ROC-AUC and classification report outputs).
- Results are interpreted in business terms (false negatives miss at-risk supporters; false positives over-allocate outreach).

## 5. Causal and Relationship Analysis
- Relationship findings are documented using feature importances and coefficient magnitudes/signs.
- Causal caution is explicit: associations are not causal proof without stronger identification design.
- Decision recommendations are tied to highest-impact features.

## 6. Deployment Notes
- Dashboard artifact is exported with `save_dashboard_json(...)` to `artifacts/dashboard_exports/`.
- Predictive model can be persisted as a joblib artifact for API inference and web app integration.
- Intended integration: CRM risk queue and stewardship prioritization view in the web application.

In [1]:
import warnings
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, mean_absolute_error, average_precision_score,
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor,
)
SEED = 42
pd.set_option("display.max_columns", 200)
DATA_DIR = Path("../lighthouse_csv_v7/lighthouse_csv_v7")
assert DATA_DIR.exists(), f"Missing data: {DATA_DIR.resolve()}"

don = pd.read_csv(DATA_DIR / "donations.csv", parse_dates=["donation_date"])
sup = pd.read_csv(DATA_DIR / "supporters.csv", parse_dates=["created_at", "first_donation_date"])
ref = don["donation_date"].max()
last_by = don.groupby("supporter_id")["donation_date"].max().rename("last_donation")
ch = sup.merge(last_by, on="supporter_id", how="left")
ch["churn"] = (ch["last_donation"].isna() | ((ref - ch["last_donation"]).dt.days > 90)).astype(int)
X = ch[["supporter_type", "relationship_type", "region", "country", "status", "acquisition_channel"]].copy()
y = ch["churn"].astype(int)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
prep = ColumnTransformer(
    [
        ("num", Pipeline([("im", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("im", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y if y.nunique() > 1 else None)
exp_m = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))])
pred_m = Pipeline([("prep", prep), ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1))])
exp_m.fit(X_tr, y_tr)
pred_m.fit(X_tr, y_tr)
proba = pred_m.predict_proba(X_te)[:, 1]
print("ROC-AUC:", roc_auc_score(y_te, proba) if y_te.nunique() > 1 else "n/a")
print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))


ROC-AUC: 0.4444444444444444
              precision    recall  f1-score   support

           0      0.500     0.333     0.400         6
           1      0.636     0.778     0.700         9

    accuracy                          0.600        15
   macro avg      0.568     0.556     0.550        15
weighted avg      0.582     0.600     0.580        15



In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pipeline_dashboard_output import export_generic_classifier_dashboard, save_dashboard_json
_fn = exp_m.named_steps["prep"].get_feature_names_out()
_coef = pd.DataFrame({"feature": _fn, "coef": exp_m.named_steps["clf"].coef_[0]})
_coef["abs_coef"] = _coef["coef"].abs()
_imp = pd.DataFrame({"feature": _fn, "importance": pred_m.named_steps["clf"].feature_importances_})
_dash = export_generic_classifier_dashboard(
    pipeline_id="donor-churn-prediction",
    display_name="Donor churn prediction",
    problem_summary="Estimates probability a supporter will not give again within 90 days (fundraising stewardship).",
    pred_m=pred_m,
    exp_m=exp_m,
    X_te=X_te,
    y_te=y_te,
    proba=proba,
    coef_df=_coef,
    imp_df=_imp,
    positive_class_description="churn (no recent gift in window)",
)
save_dashboard_json(_dash)
print("dashboard_export:", _dash.get("export_path"))


dashboard_export: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\dashboard_exports\donor-churn-prediction.json


## Note
Improve with rolling donation frequency features per supporter. **Deployment:** CRM churn list and stewardship cadence.


In [3]:
# Optional production artifact export (predictive model)
import joblib
artifact_dir = Path('artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / 'donor_churn_prediction_rf.joblib'
joblib.dump(pred_m, artifact_path)
print('saved:', artifact_path.resolve())

saved: C:\Users\hoope\OneDrive\Desktop\ml-pipelines\ml-pipelines\artifacts\donor_churn_prediction_rf.joblib
